# 🧠 Auto-Hyperontology (`Hyper-RAG`) — Cuaderno Maestro Completo
**Compilación con Modelo Docente MoE 125B (`qwen3.8-flash-next:125b-a6b-q4_K_M`) y Razonamiento Proto-AGI (`intuyt:macro`)**

**Autor:** [Dr. Christian Efraín Maldonado-Sifuentes](https://github.com/christianemaldonadomti) (CIC-IPN / SECIHTI / TRAI-L)

Este cuaderno interactivo ejecuta la compilación de hiperontologías $n$-arias aprovechando un modelo de alta capacidad como **Docente** (`qwen3.8-flash-next:125b-a6b-q4_K_M`) a través de tu túnel de ngrok, persistiendo el resultado en Google Drive para razonamiento ultrarrápido con **`intuyt:macro`**.

--- 
## Paso 1: Instalación de Dependencias (Sin Conflictos con Colab)

In [ ]:
# Usamos --no-deps para evitar alterar pandas/requests preinstalados en Google Colab
!pip install -q --no-deps git+https://github.com/christianemaldonadomti/auto-hyperontology.git
!pip install -q datasets

--- 
## Paso 2: Conexión con Ollama (`qwen3.8-flash-next:125b-a6b-q4_K_M` en ngrok)
Configuramos el modelo docente de 125B para la extracción formal de hiperaristas $n$-arias con un `timeout` de 180 segundos para inferencia profunda.

In [ ]:
import requests
from auto_hyperontology import HyperRAG, OllamaClient, HyperEdge

NGROK_ENDPOINT = "https://comic-hugely-eft.ngrok-free.app"
EXTRACTION_MODEL = "qwen3.8-flash-next:125b-a6b-q4_K_M"
REASONING_MODEL = "intuyt:macro"  # Modelo ligero para la deducción en 5 etapas
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

# Función normalizadora defensiva (garantiza que node_weights y roles sean diccionarios)
def sanitize_edge(edge: HyperEdge):
    if not isinstance(edge.nodes, list):
        edge.nodes = [str(edge.nodes)]
    n_len = max(1, len(edge.nodes))
    unif = round(1.0 / n_len, 4)
    if isinstance(edge.node_weights, list):
        edge.node_weights = {
            nid: (float(edge.node_weights[i]) if i < len(edge.node_weights) and isinstance(edge.node_weights[i], (int, float)) else unif)
            for i, nid in enumerate(edge.nodes)
        }
    elif not isinstance(edge.node_weights, dict):
        edge.node_weights = {nid: unif for nid in edge.nodes}
    if not isinstance(edge.roles, dict):
        edge.roles = {f"role_{i+1}": nid for i, nid in enumerate(edge.nodes)}
    if not isinstance(edge.attributes, dict):
        edge.attributes = {"description": str(edge.attributes)}
    return edge

# Verificamos disponibilidad del servidor y si el modelo 125B está cargado en Ollama
try:
    resp = requests.get(f"{NGROK_ENDPOINT}/api/tags", headers=HEADERS, timeout=10)
    if resp.status_code == 200:
        available_models = [m.get("name") for m in resp.json().get("models", [])]
        print(f"✅ Conectado exitosamente al servidor Ollama en: {NGROK_ENDPOINT}")
        print(f"📦 Modelos disponibles en el nodo: {available_models}")
        if any(EXTRACTION_MODEL.lower() in m.lower() for m in available_models):
            print(f"🎯 Modelo docente detectado: {EXTRACTION_MODEL}")
        else:
            print(f"ℹ️ Aviso: '{EXTRACTION_MODEL}' no aparece listado en tags; asegúrate de haber hecho 'ollama run {EXTRACTION_MODEL}' en tu servidor local.")
    else:
        print(f"⚠️ El endpoint respondió con código HTTP {resp.status_code}.")
except Exception as e:
    print(f"ℹ️ Servidor ngrok no disponible ({e}).")

--- 
## Paso 3: Persistencia en Google Drive (Carga Instantánea o Guardado)
Si la hiperontología ya fue extraída con `qwen3.8-flash-next:125b-a6b-q4_K_M`, se carga en **1 segundo**. Si no, se compila y se guarda.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Hyperontology_Cache'
os.makedirs(DRIVE_DIR, exist_ok=True)
CACHE_FILE = os.path.join(DRIVE_DIR, 'qwen125b_pubmed_hyperontology.json')

rag = None
if os.path.exists(CACHE_FILE):
    print(f"⚡ Encontrada hiperontología previa de 125B en Google Drive: {CACHE_FILE}")
    print(f"   Tamaño del archivo: {os.path.getsize(CACHE_FILE) / 1024:.2f} KB")
    rag = HyperRAG.load(CACHE_FILE, model=REASONING_MODEL, base_url=NGROK_ENDPOINT)
    rag.client.headers = HEADERS
    print("✅ Hiperontología cargada instantáneamente desde Google Drive (extracción de 125B omitida).")
else:
    print("ℹ️ No se encontró caché previo de 125B en Google Drive. Se procederá a compilar en los siguientes pasos.")

--- 
## Paso 4: Ingesta de Dataset desde Hugging Face (`PubMedQA`)

In [ ]:
from datasets import load_dataset

print("📥 Descargando subconjunto de PubMedQA desde Hugging Face...")
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train[:6]")

pubmed_samples = []
for idx, row in enumerate(ds):
    ctx_str = " ".join(row["context"]["contexts"])
    pubmed_samples.append({
        "id": f"PubMedQA_{idx+1}",
        "question": row["question"],
        "context": ctx_str,
        "long_answer": row["long_answer"],
        "final_decision": row["final_decision"]
    })

print(f"✅ Se prepararon {len(pubmed_samples)} artículos clínicos para análisis.")
print("Ejemplo de artículo:\n", pubmed_samples[0]["context"][:320], "...")

--- 
## Paso 5: Compilación de la Hiperontología con `qwen3.8-flash-next:125b-a6b-q4_K_M` y Guardado en Drive

In [ ]:
if rag is None:
    print(f"⚙️ Compilando hiperontología usando el modelo docente: {EXTRACTION_MODEL}...")
    rag = HyperRAG(
        model=EXTRACTION_MODEL,
        base_url=NGROK_ENDPOINT,
        default_domain="PubMed_Clinical",
        theta_max=3000
    )
    # Inyectamos headers y configuramos timeout amplio (180s) para el modelo 125B
    rag.client.headers = HEADERS
    rag.client.timeout = 180
    rag.extractor.client.headers = HEADERS
    rag.extractor.client.timeout = 180
    
    for s in pubmed_samples:
        print(f"   -> Procesando documento {s['id']}...")
        rag.add_text(s["context"], domain="PubMed_Clinical")
    
    # Normalizamos hiperaristas generadas
    for e in rag.hyper.edges.values():
        sanitize_edge(e)
    rag._refresh_engine()
    
    # Guardamos en Google Drive para sesiones futuras
    rag.save(CACHE_FILE)
    print(f"💾 Hiperontología compilada con 125B guardada exitosamente en Google Drive:\n   {CACHE_FILE}")
else:
    print("⚡ Usando hiperontología existente de 125B cargada desde Google Drive.")

# Mostramos estadísticas topológicas
import json
print("\n=== ESTADÍSTICAS TOPOLÓGICAS DE LA HIPERONTOLOGÍA ===")
print(json.dumps(rag.stats(), indent=2, ensure_ascii=False))

--- 
## Paso 6: Inspección y Auditoría Visual de la Hiperontología Extraída por 125B

In [ ]:
import pandas as pd
from auto_hyperontology import inspect_hyperontology

audit = inspect_hyperontology(rag.hyper)

print(f"📊 Vértices totales |X|: {audit['total_vertices']}")
print(f"📊 Hiperaristas n-arias |E|: {audit['total_hyperedges']}")
print(f"📊 Espectro de Cardinalidad (|H_e|): {audit['cardinality_spectrum']}")
print(f"📊 Distribución de Categorías: {audit['categories_distribution']}")

# Tabla detallada de hiperaristas extraídas por el modelo 125B
df_edges = pd.DataFrame([
    {
        "ID": e["edge_id"],
        "Tipo": e["relation_type"],
        "Cardinalidad": f"{e['cardinality']}-aria",
        "Capa RAPTOR": f"L_{e['raptor_layer']}",
        "Vértices Participantes": ", ".join(e["participating_nodes"][:5]),
        "Evidencia": e["evidence"][:90] + "..." if e["evidence"] else ""
    }
    for e in audit["edges"]
])
display(df_edges.head(15))

# Nodos Frontera (puentes entre subdominios)
df_boundary = pd.DataFrame(audit["boundary_nodes"])
if not df_boundary.empty:
    print("\n⚓ NODOS FRONTERA DETECTADOS:")
    display(df_boundary)
else:
    print("\nℹ️ Todos los vértices pertenecen al dominio clínico analizado.")

--- 
## Paso 7: Grafo Topológico Interactivo en Vivo (Directo en Colab)

In [ ]:
import IPython
from auto_hyperontology import generate_interactive_html

html_path = generate_interactive_html(rag.hyper, "hyperontology_interactive.html")
print(f"🌐 Visualizador interactivo generado en: {html_path}")
IPython.display.display(IPython.display.HTML(filename=html_path))

--- 
## Paso 8: Consulta de Razonamiento Proto-AGI en 5 Etapas
Consultamos la hiperontología compilada usando `intuyt:macro` (o el modelo configurado).

In [ ]:
for e in rag.hyper.edges.values():
    sanitize_edge(e)

# Configuramos el cliente del razonador hacia intuyt:macro para deducción rápida y ecológica
rag.client.model = REASONING_MODEL
rag.reasoner.client.model = REASONING_MODEL

query_text = pubmed_samples[0]["question"]
result = rag.query(query_text, max_edges=4)

print(f"🔎 PREGUNTA: {result.question}\n")
print(f"🚪 Compuerta RAPTOR: {result.raptor_gate_info}")
print(f"⚓ Nodos Semilla Activados: {result.seed_nodes}")
print(f"📐 Puntajes F_OC (OEIS A320228): {result.f_oc_scores}\n")
print(result.minimal_response_context)
print("\n" + "="*75)
print(f"🧠 RESPUESTA GENERADA ({REASONING_MODEL} + Proto-AGI):")
print("="*75)
print(result.answer)
print("\n" + "="*75)
print("🌱 MÉTRICAS AME & HUELLA ECOLÓGICA GREEN-AGI:")
print("="*75)
print(f"• Ahorro de Contexto (A): {result.metrics.savings_percent}% ({result.metrics.context_tokens_used} tokens vs {result.metrics.context_tokens_baseline} baseline)")
print(f"• Energía Consumida (3B MoE): {result.metrics.energy_wh_3b:.6f} Wh vs {result.metrics.energy_wh_400b_baseline:.4f} Wh de un modelo 400B (~{result.metrics.compute_reduction_factor}x reducción)")
print(f"• Emisiones de Carbono: {result.metrics.co2_kg_3b:.8f} kg CO2")

--- 
## Paso 9: Benchmark Head-to-Head Real (`Hyper-RAG` vs. `Okapi BM25 Flat Text RAG`)

In [ ]:
import re, math, time
import pandas as pd
import matplotlib.pyplot as plt
from auto_hyperontology.extractor import estimate_bpe_tokens

class RealBM25FlatRAG:
    def __init__(self, client, chunk_words=75, overlap_words=15, k1=1.5, b=0.75):
        self.client = client
        self.chunk_words, self.overlap_words = chunk_words, overlap_words
        self.k1, self.b = k1, b
        self.chunks, self.chunk_tokens, self.doc_freqs = [], [], {}
        self.avgdl = 1.0

    @staticmethod
    def _tok(text):
        return [w.lower() for w in re.findall(r"[a-zA-Z0-9]{3,}", text)]

    def ingest(self, docs):
        step = max(1, self.chunk_words - self.overlap_words)
        for d in docs:
            words = d.split()
            if len(words) <= self.chunk_words:
                self.chunks.append(d.strip())
            else:
                for i in range(0, len(words), step):
                    c = " ".join(words[i:i + self.chunk_words]).strip()
                    if c: self.chunks.append(c)
        self.chunk_tokens = [self._tok(c) for c in self.chunks]
        for toks in self.chunk_tokens:
            for ut in set(toks):
                self.doc_freqs[ut] = self.doc_freqs.get(ut, 0) + 1
        self.avgdl = sum(len(t) for t in self.chunk_tokens) / float(max(1, len(self.chunk_tokens)))

    def retrieve(self, query, top_k=4):
        q_toks = self._tok(query)
        n_docs = len(self.chunks)
        scores = []
        for idx, c_toks in enumerate(self.chunk_tokens):
            dl = len(c_toks)
            sc = 0.0
            for qt in q_toks:
                df = self.doc_freqs.get(qt, 0)
                if df == 0: continue
                idf = math.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
                tf = c_toks.count(qt)
                sc += idf * ((tf * (self.k1 + 1.0)) / (tf + self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))))
            scores.append((idx, round(sc, 4), self.chunks[idx]))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

    def query(self, question, top_k=4):
        t0 = time.perf_counter()
        top_c = self.retrieve(question, top_k=top_k)
        ctx_str = "\n\n".join(f"[Chunk {i} | BM25={s}]: {t}" for i, s, t in top_c)
        prompt = f"Context Chunks:\n{ctx_str}\n\nQuestion: {question}\nAnswer strictly based on the context above:"
        ans = self.client.generate(prompt)
        return {"answer": ans, "context": ctx_str, "tokens": estimate_bpe_tokens(prompt), "latency_ms": round((time.perf_counter() - t0)*1000, 1)}

STOPWORDS = {"the","and","for","that","this","with","from","are","was","were","have","has","not","but","what","which","when","where","how","why","can","could","would","should","based","context","chunks","hyperedge","hyperontology","proof","subgraph","question","answer","there","their","they","been","into","more","than","also","such","only","other","some","these","those","between","through","during","after","before","under","does","did"}

def content_words(text):
    return {w for w in re.findall(r"[a-zA-Z]{4,}", text.lower()) if w not in STOPWORDS}

def evaluate_live_answer(question, generated_answer, source_context, ground_truth_answer):
    ans_w = content_words(generated_answer)
    ctx_w = content_words(source_context + " " + question)
    gt_w = content_words(ground_truth_answer)
    if not ans_w:
        return 0.0, 0.0, 100.0
    supported = len(ans_w.intersection(ctx_w)) / float(len(ans_w))
    ungrounded = 1.0 - supported
    recall = len(ans_w.intersection(gt_w)) / float(max(1, len(gt_w)))
    return round(supported * 100, 1), round(recall * 100, 1), round(ungrounded * 100, 1)

live_client = OllamaClient(base_url=NGROK_ENDPOINT, model=REASONING_MODEL, temperature=0.1)
live_client.headers = HEADERS

flat_rag = RealBM25FlatRAG(client=live_client, chunk_words=75, overlap_words=15)
flat_rag.ingest([s["context"] for s in pubmed_samples])

rows = []
for s in pubmed_samples:
    for e in rag.hyper.edges.values():
        sanitize_edge(e)
    t0 = time.perf_counter()
    h_out = rag.query(s["question"], max_edges=4)
    h_lat = round((time.perf_counter() - t0)*1000, 1)
    f_out = flat_rag.query(s["question"], top_k=4)

    h_faith, h_rec, h_hal = evaluate_live_answer(s["question"], h_out.answer, s["context"] + " " + h_out.minimal_response_context, s["long_answer"])
    f_faith, f_rec, f_hal = evaluate_live_answer(s["question"], f_out["answer"], s["context"], s["long_answer"])

    rows.append({
        "ID": s["id"],
        "Hyper Faithfulness (%)": h_faith,
        "Flat Faithfulness (%)": f_faith,
        "Hyper Hallucination (%)": h_hal,
        "Flat Hallucination (%)": f_hal,
        "Hyper GT Recall (%)": h_rec,
        "Flat GT Recall (%)": f_rec,
        "Hyper Tokens": h_out.metrics.context_tokens_used,
        "Flat Tokens": f_out["tokens"],
        "Hyper Latency (ms)": h_lat,
        "Flat Latency (ms)": f_out["latency_ms"]
    })

df_bench = pd.DataFrame(rows)
display(df_bench)

print("\n=== PROMEDIOS EN VIVO (PubMedQA + intuyt:macro) ===")
print(df_bench.mean(numeric_only=True).round(2))

# Gráfica de barras comparativa
summary_plot = pd.DataFrame({
    "Métrica": ["Faithfulness (%) ↑", "Tokens no Fundamentados (%) ↓", "Recall de Conceptos (%) ↑", "Tokens de Prompt Promedio ↓"],
    "Flat Text RAG (BM25)": [
        df_bench["Flat Faithfulness (%)"].mean(),
        df_bench["Flat Hallucination (%)"].mean(),
        df_bench["Flat GT Recall (%)"].mean(),
        df_bench["Flat Tokens"].mean()
    ],
    "Hyper-RAG (Ours)": [
        df_bench["Hyper Faithfulness (%)"].mean(),
        df_bench["Hyper Hallucination (%)"].mean(),
        df_bench["Hyper GT Recall (%)"].mean(),
        df_bench["Hyper Tokens"].mean()
    ]
})
ax = summary_plot.plot(x="Métrica", kind="bar", figsize=(10, 5), rot=15, title="Evaluación en Vivo sobre PubMedQA: Hyper-RAG vs. Okapi BM25 Flat Text RAG")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

--- 
## Paso 10: Exportación de Reificación Topológica WebGL (`reader-math.php`)

In [ ]:
webgl_file = "hiperontologia_final.json"
webgl_data = rag.export_webgl_reification(webgl_file, n_limit=300)
print(f"✅ Subgrafo exportado exitosamente a '{webgl_file}'")
print(f"   Hub Central: {webgl_data['hub']} | Elementos Renderizados: {webgl_data['total_rendered']}")
print(f"   Enlaces Dirigidos (v_s -> R_k -> v_t): {len(webgl_data['links'])}")